## **Statistical Language Modeling: The "Lightweight" Autocomplete Engine**

You have been hired as a Junior NLP Engineer by a mobile software startup. The company is developing a keyboard application for low-resource feature phones that cannot run heavy Deep Learning models (like Transformers).


Your task is to build a lightweight Predictive Text Engine using statistical N-Grams. The system must learn from a corpus of text, handle words it hasn't seen frequently (Smoothing), generate coherent sentence completions, and mathematically prove its accuracy (Perplexity).


In [1]:
# Importing Required Libraries
import re
from collections import Counter, defaultdict
import random
import math
import requests
import os

#### **Task-1: N-Gram Extraction**

#### What is an N-Gram?

An N-Gram is a contiguous sequence of N items (words, characters, etc.) from a given text.

For example, in the sentence "I love machine learning", the 2-grams (bigrams) are:

["I love", "love machine", "machine learning"]

#### N-Gram Probability Equation
The probability of a word \( w_n \) given its history \( h \) in an n-gram model is:
$$
P(w_n \mid h) = \frac{C(h, w_n)}{\sum_{w'} C(h, w')}
$$
Where:
- \( C(h, w_n) \): Count of occurrences of \( h \) followed by \( w_n \)
- \( h \): Context (history of n-1 words)

<br>
<br>

##### **Task:**
1. Write a function `generate_ngrams(text, n)` that takes a corpus of text and breaks it down into contiguous sequences of `N` items.
```
Input: "I love machine learning"
Character-Level Bigrams: [('#', 'I'), ('I', ' '), (' ', 'l'), ('l', 'o'), ('o', 'v'), ('v', 'e'), ('e', ' '), (' ', 'm'), ('m', 'a'), ('a', 'c'), ('c', 'h'), ('h', 'i'), ('i', 'n'), ('n', 'e'), ('e', ' '), (' ', 'l'), ('l', 'e'), ('e', 'a'), ('a', 'r'), ('r', 'n'), ('n', 'i'), ('i', 'n'), ('n', 'g')]

```

2. Build an n-gram language model from the corpus. Write a function `build_ngram_model(corpus, n)` that takes the corpus of text and breaks it down into contiguous `N` items, and return a probability distribution for each context.

```
Input: "I love machine learning"
output: {('#',): Counter({'I': 1.0}),
             ('I',): Counter({' ': 1.0}),
             (' ',): Counter({'l': 0.6666666666666666,
                      'm': 0.3333333333333333}),
             ('l',): Counter({'o': 0.5, 'e': 0.5}),
             ('o',): Counter({'v': 1.0}),
             ('v',): Counter({'e': 1.0}),
             ('e',): Counter({' ': 0.6666666666666666,
                      'a': 0.3333333333333333}),
             ('m',): Counter({'a': 1.0}),
             ('a',): Counter({'c': 0.5, 'r': 0.5}),
             ('c',): Counter({'h': 1.0}),
             ('h',): Counter({'i': 1.0}),
             ('i',): Counter({'n': 1.0}),
             ('n',): Counter({'e': 0.3333333333333333,
                      'i': 0.3333333333333333,
                      'g': 0.3333333333333333}),
             ('r',): Counter({'n': 1.0})}
```

In [2]:
def generate_ngrams(text, n):
    # Add a start token to handle the beginning of the text
    processed_text = '#' * (n - 1) + text
    ngrams = []
    for i in range(len(processed_text) - n + 1):
        ngrams.append(tuple(processed_text[i:i+n]))
    return ngrams

In [3]:
# Example Text
text = "I love machine learning"

# Generate and display bigrams (2-grams)
bigrams = generate_ngrams(text, 2)
print("Character-Level Bigrams:", bigrams)

Character-Level Bigrams: [('#', 'I'), ('I', ' '), (' ', 'l'), ('l', 'o'), ('o', 'v'), ('v', 'e'), ('e', ' '), (' ', 'm'), ('m', 'a'), ('a', 'c'), ('c', 'h'), ('h', 'i'), ('i', 'n'), ('n', 'e'), ('e', ' '), (' ', 'l'), ('l', 'e'), ('e', 'a'), ('a', 'r'), ('r', 'n'), ('n', 'i'), ('i', 'n'), ('n', 'g')]


In [4]:
def build_ngram_model(corpus, n):
    # Use defaultdict to store counts of (context, next_char)
    ngram_counts = defaultdict(lambda: defaultdict(int))

    # Add start tokens to the corpus for n-gram generation
    processed_text = '#' * (n - 1) + corpus

    # Collect vocabulary from the processed text (including '#')
    vocabulary_set = set(processed_text)

    # Generate n-grams and populate ngram_counts
    for i in range(len(processed_text) - n + 1):
        current_ngram_tuple = tuple(processed_text[i:i+n])
        context = current_ngram_tuple[:-1]  # The first n-1 characters form the context
        next_char = current_ngram_tuple[-1] # The last character is the predicted character
        ngram_counts[context][next_char] += 1

    # Return raw counts and the vocabulary set
    return ngram_counts, vocabulary_set

In [5]:
# Sample Test Corpus
corpus = "I love machine learning"

# Call the refactored build_ngram_model, which now returns raw counts and the vocabulary set
raw_ngram_counts, vocabulary_set = build_ngram_model(corpus, 2)

print("Raw N-gram Counts (before smoothing/probability calculation):", raw_ngram_counts)
print("Vocabulary Set:", vocabulary_set)

# The expected output from the problem description shows a probability model with start tokens.
# The next step, `add_smoothing`, will convert these raw counts into smoothed probabilities,
# matching the spirit of the expected output, but with smoothing applied.
# The original `build_ngram_model` re-definition and its test are removed for consistency.

Raw N-gram Counts (before smoothing/probability calculation): defaultdict(<function build_ngram_model.<locals>.<lambda> at 0x7a2288bdfa60>, {('#',): defaultdict(<class 'int'>, {'I': 1}), ('I',): defaultdict(<class 'int'>, {' ': 1}), (' ',): defaultdict(<class 'int'>, {'l': 2, 'm': 1}), ('l',): defaultdict(<class 'int'>, {'o': 1, 'e': 1}), ('o',): defaultdict(<class 'int'>, {'v': 1}), ('v',): defaultdict(<class 'int'>, {'e': 1}), ('e',): defaultdict(<class 'int'>, {' ': 2, 'a': 1}), ('m',): defaultdict(<class 'int'>, {'a': 1}), ('a',): defaultdict(<class 'int'>, {'c': 1, 'r': 1}), ('c',): defaultdict(<class 'int'>, {'h': 1}), ('h',): defaultdict(<class 'int'>, {'i': 1}), ('i',): defaultdict(<class 'int'>, {'n': 2}), ('n',): defaultdict(<class 'int'>, {'e': 1, 'i': 1, 'g': 1}), ('r',): defaultdict(<class 'int'>, {'n': 1})})
Vocabulary Set: {'v', 'n', 'h', 'I', 'g', 'r', 'm', 'o', '#', 'e', ' ', 'i', 'c', 'a', 'l'}


#### **Task 2: Handling Sparsity (Smoothing)**
Real-world data is sparse. If a user types a word combination not found in your training data, your model returns a probability of 0, crashing the system. Implement Add-$\alpha $ (Laplace) Smoothing.

Smoothing assigns a small non-zero probability to unseen n-grams.

#### Smoothing Equation
With smoothing, the probability becomes:
$$
P(w_n \mid h) = \frac{C(h, w_n) + \alpha}{\sum_{w'} C(h, w') + \alpha \times |V|}
$$
Where:
- $\alpha $: Smoothing parameter (default is 1)
- \( |V| \): Vocabulary size


In [6]:
def add_smoothing(ngram_counts, vocabulary_set, alpha=1.0):
    # Initialize the smoothed model to store probabilities for each context
    smoothed_model = defaultdict(Counter)
    vocabulary_size = len(vocabulary_set)

    # Iterate through each observed context in the raw n-gram counts
    for context in ngram_counts.keys():
        char_counts = ngram_counts[context]
        total_count_for_context = sum(char_counts.values())

        # Calculate the denominator for the smoothing formula
        denominator = total_count_for_context + alpha * vocabulary_size

        # Apply smoothing for every character in the global vocabulary
        for char_in_vocab in vocabulary_set:
            # Get the count for the character in the current context (0 if not seen)
            count_of_char_in_context = char_counts[char_in_vocab]

            # Calculate the smoothed probability
            smoothed_prob = (count_of_char_in_context + alpha) / denominator
            smoothed_model[context][char_in_vocab] = smoothed_prob

    # Note: This function only smooths probabilities for contexts *observed* in the training data.
    # If a completely unseen context is encountered during text generation,
    # it would typically be handled by assigning a uniform distribution (1/vocabulary_size)
    # or backing off to a lower-order model.

    return smoothed_model

#### **Task 3: Text Generation**
Create a function `generate_text(context, length)`:
1. Take a starting word/phrase (context).
2. Look up the probabilities of all possible next words.
3. Sample the next word based on these probabilities.
4. Update the context and repeat until the desired length is reached.



In [9]:
def generate_text(model, n, start_text, vocabulary_set, length=100):
    # Ensure start_text is a string
    start_text_str = str(start_text)

    # Initialize generated_sequence with start_text
    generated_sequence = list(start_text_str)

    # Generate characters until the desired length is reached
    for _ in range(length):
        # The context for predicting the next character is the last (n-1) characters
        # from the *currently generated* sequence, potentially padded with '#' if too short.
        current_context_list = generated_sequence[-(n - 1):] if (n - 1) > 0 else []

        if len(current_context_list) < (n - 1):
            # Pad with '#' if current sequence is too short to form a full context
            padding_needed = (n - 1) - len(current_context_list)
            current_context_list = ['#'] * padding_needed + current_context_list

        current_context = tuple(current_context_list)

        # Get probabilities for the next character
        if current_context in model:
            char_probabilities = model[current_context]
            # Extract characters and their probabilities
            choices = list(char_probabilities.keys())
            weights = list(char_probabilities.values())
        else:
            # If the context is completely unseen, distribute probability uniformly
            # among all characters in the vocabulary
            choices = list(vocabulary_set)
            weights = [1.0 / len(vocabulary_set)] * len(vocabulary_set) # Assign uniform probability

        # Sample the next character based on weights
        if not choices: # Should not happen if vocabulary_set is not empty
            next_char = random.choice(list(vocabulary_set)) # Fallback
        else:
            next_char = random.choices(choices, weights=weights, k=1)[0]

        generated_sequence.append(next_char)

    return "".join(generated_sequence)


In [10]:
# Sample text
text = "hello world this is a sample text for testing the n-gram model"

# Build a bigram model (raw counts and vocabulary)
raw_ngram_counts, vocabulary_set = build_ngram_model(text, 2)

# Apply smoothing to get the probability model
smoothed_model = add_smoothing(raw_ngram_counts, vocabulary_set, alpha=1.0)

# Generate text using the smoothed model and vocabulary
generated = generate_text(smoothed_model, 2, "he", vocabulary_set, 10)
print(f"Generated text: {generated}")

Generated text: hemgnhele ne


#### **Task 4: Model Evaluation (Perplexity)**
You need to prove to your manager that your model is actually learning. Implement the Perplexity metric on a held-out test set. Lower perplexity indicates a better model.

Action: Calculate the perplexity of your model on a short unseen test sentence (e.g., "The machine learning algorithm").

#### **Perplexity**

Perplexity is a common metric for evaluating language models. Lower perplexity indicates a better model.

#### Perplexity Equation
Perplexity measures how well a language model predicts a test dataset:
$$
PP(W) = 2^{-\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i \mid h_i)}
$$
Where:
- \( W \): Sequence of words
- \( N \): Total number of words in the sequence
- $ P(w_i \mid h_i) $: Probability of word $w_i$ given its history $ h_i $

In [33]:
def calculate_perplexity(model, n, test_text, vocabulary_set, alpha=1.0):
    log_sum_prob = 0.0
    N = 0
    padded_text = ['#'] * (n - 1) + list(test_text)

    for i in range(n - 1, len(padded_text)):
        context = tuple(padded_text[i - (n - 1):i])
        word = padded_text[i]
        con_prob = model.get(context, {})
        if context not in model:
            V = len(vocabulary_set) # Corrected from 'vocabulary'
            alpha = 1.0
            prob = alpha / (alpha * V)
        else:
            prob = con_prob.get(word, 0.0)
        if prob == 0:
            return float('inf')

        log_sum_prob += math.log2(prob)
        N += 1

    if N == 0:
        return float('inf')

    perplexity = 2 ** (-log_sum_prob / N)
    return perplexity

In [23]:
# First, let's create a more substantial training corpus
training_corpus = """
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
How much wood would a woodchuck chuck if a woodchuck could chuck wood?
To be or not to be, that is the question.
All that glitters is not gold.
A journey of a thousand miles begins with a single step.
Actions speak louder than words.
Beauty is in the eye of the beholder.
Every cloud has a silver lining.
Fortune favors the bold and brave.
Life is like a box of chocolates.
The early bird catches the worm.
Where there's smoke, there's fire.
Time heals all wounds and teaches all things.
Knowledge is power, and power corrupts.
Practice makes perfect, but nobody's perfect.
The pen is mightier than the sword.
When in Rome, do as the Romans do.
A picture is worth a thousand words.
Better late than never, but never late is better.
Experience is the best teacher of all things.
Laughter is the best medicine for the soul.
Music soothes the savage beast within us.
Nothing ventured, nothing gained in life.
The grass is always greener on the other side.
"""

# Clean the corpus
training_corpus = ''.join(c.lower() for c in training_corpus if c.isalnum() or c.isspace())

In [24]:
training_corpus

'\nthe quick brown fox jumps over the lazy dog\nshe sells seashells by the seashore\nhow much wood would a woodchuck chuck if a woodchuck could chuck wood\nto be or not to be that is the question\nall that glitters is not gold\na journey of a thousand miles begins with a single step\nactions speak louder than words\nbeauty is in the eye of the beholder\nevery cloud has a silver lining\nfortune favors the bold and brave\nlife is like a box of chocolates\nthe early bird catches the worm\nwhere theres smoke theres fire\ntime heals all wounds and teaches all things\nknowledge is power and power corrupts\npractice makes perfect but nobodys perfect\nthe pen is mightier than the sword\nwhen in rome do as the romans do\na picture is worth a thousand words\nbetter late than never but never late is better\nexperience is the best teacher of all things\nlaughter is the best medicine for the soul\nmusic soothes the savage beast within us\nnothing ventured nothing gained in life\nthe grass is always

In [28]:
# Build models of different orders (bigram, trigram, and 4-gram models)
def build_models(corpus):
    models = {}
    for n in [2, 3, 4]:
        # Unpack the tuple returned by build_ngram_model
        raw_ngram_counts, vocabulary_set = build_ngram_model(corpus, n)

        # Now pass the correct raw_ngram_counts and vocabulary_set to add_smoothing
        smoothed_model = add_smoothing(raw_ngram_counts, vocabulary_set, alpha=1.0)

        # Store both the smoothed model and its vocabulary_set
        models[n] = {'model': smoothed_model, 'vocabulary': vocabulary_set}
    return models

# Build the models
models = build_models(training_corpus)

In [30]:
# Generate samples and calculate perplexity
def evaluate_samples(models, num_samples=10, sample_length=40):
    """
    Evaluates multiple n-gram language models by generating text samples and calculating their perplexity scores.

    This function:
    1. Takes different n-gram models (e.g., bigram, trigram, 4-gram)
    2. For each model:
       - Generates multiple text samples
       - Calculates perplexity for each sample
       - Computes average perplexity across all samples
    3. Stores and returns all results for comparison

    Parameters:
    -----------
    models : dict
        Dictionary where keys are n-gram sizes (e.g., 2 for bigram)
        and values are dictionaries containing {'model': smoothed_model, 'vocabulary': vocabulary_set}

    num_samples : int, optional (default=10)
        Number of text samples to generate for each model

    sample_length : int, optional (default=40)
        Length of each generated text sample in characters

    Returns:
    --------
    dict
        A dictionary where:
        - Keys are n-gram sizes (2, 3, 4, etc.)
        - Values are lists of dictionaries containing:
          {'text': generated_text, 'perplexity': perplexity_score}

    Example:
    --------
    # Example output structure:
    {
        2: [  # Results for bigram model
            {'text': 'hello world', 'perplexity': 10.5},
            {'text': 'sample text', 'perplexity': 12.3},
            # ... more samples
        ],
        3: [  # Results for trigram model
            {'text': 'another example', 'perplexity': 8.7},
            # ... more samples
        ]
    }
    """
    results = defaultdict(list)
    for n, model_data in models.items():
        current_model = model_data['model']
        vocabulary = model_data['vocabulary']
        print(f"\nEvaluating {n}-gram model...")
        for i in range(num_samples):
            start_char = random.choice(list(vocabulary))
            generated_text = generate_text(current_model, n, start_char, vocabulary, length=sample_length)
            perplexity_score = calculate_perplexity(current_model, n, generated_text, vocabulary)

            results[n].append({'text': generated_text, 'perplexity': perplexity_score})
            print(f"  Sample {i+1}: '{generated_text[:20]}...' Perplexity: {perplexity_score:.2f}")
    return results

In [34]:
# Evaluate samples
results = evaluate_samples(models)

# Calculate statistics for each model
print("\n=== Overall Statistics ===")
for n in models.keys():
    perplexities = [sample['perplexity'] for sample in results[n]]
    min_perp = min(perplexities)
    max_perp = max(perplexities)
    avg_perp = sum(perplexities) / len(perplexities)

    print(f"\n{n}-gram Model Statistics:")
    print(f"Minimum Perplexity: {min_perp:.2f}")
    print(f"Maximum Perplexity: {max_perp:.2f}")
    print(f"Average Perplexity: {avg_perp:.2f}")


Evaluating 2-gram model...
  Sample 1: 'cwajldxqxitpnwin gyv...' Perplexity: 27.19
  Sample 2: 'fothe
xg
ti#vetthebg...' Perplexity: 14.86
  Sample 3: 'tczkv#krofzydus utco...' Perplexity: 22.06
  Sample 4: 'rfwin worfi lstthe e...' Perplexity: 14.64
  Sample 5: 'qd wfbzve olil wo ie...' Perplexity: 20.69
  Sample 6: '
akr we th#dfbueys w...' Perplexity: 17.87
  Sample 7: 'fothodo lv
ld
k  cks...' Perplexity: 18.48
  Sample 8: 'ath uf wtgxwe ges
jd...' Perplexity: 19.78
  Sample 9: '#dode d win sphiqoyp...' Perplexity: 17.70
  Sample 10: 'ct#bmedgnkul w
hthqh...' Perplexity: 23.74

Evaluating 3-gram model...
  Sample 1: 'lszvnogbvaummuckwwoo...' Perplexity: 25.32
  Sample 2: 'loqhbtbhmocbgkn#t
 v...' Perplexity: 29.47
  Sample 3: 'ssvjqpxr
t#aytpkt#tt...' Perplexity: 29.07
  Sample 4: '
hdfeaujfqkq  pimhxn...' Perplexity: 28.22
  Sample 5: '
axmtm##jfkkxbetswsp...' Perplexity: 26.15
  Sample 6: 'uibc
xwkl rolydikpra...' Perplexity: 28.08
  Sample 7: ' rz icrcxdkyjpvpamqx...' Perplexit

#### **Deliverables**
1. A Python Notebook (.ipynb) containing the implementations of the equations above.
2. A brief analysis report (100 words) answering: How did changing N (e.g., from Bigram to Trigram) affect the coherence of the generated text and the Perplexity score?